In [4]:
import joblib
import pandas as pd

# Load the saved model and label encoder
loaded_model_pipeline = joblib.load('crop_prediction_model.pkl')
loaded_label_encoder = joblib.load('crop_label_encoder.pkl')

print("Model and label encoder loaded successfully.")

# Define new input data for prediction
# You can modify these values to test different conditions
new_data = pd.DataFrame([
    {
        'Nitrogen': 0,
        'Phosphorus': 10,
        'Potassium': 0,
        'Temperature': 20,
        'Humidity': 20,
        'pH_Value': 7.5,
        'Rainfall': 10,
        'Soil_Type': '',
        'Variety': ''
    }
])

# Make a prediction using the loaded model
predicted_crop_encoded = loaded_model_pipeline.predict(new_data)

# Decode the predicted crop back to its original name
predicted_crop_name = loaded_label_encoder.inverse_transform(predicted_crop_encoded)

print(f"\nInput Conditions:\n{new_data.to_string(index=False)}")
print(f"\nPredicted Crop: {predicted_crop_name[0]}")

Model and label encoder loaded successfully.

Input Conditions:
 Nitrogen  Phosphorus  Potassium  Temperature  Humidity  pH_Value  Rainfall Soil_Type Variety
        0          10          0           20        20       7.5        10                  

Predicted Crop: Tomato


In [12]:
def get_optimal_ranges(crop_name, dataframe):
    """
    Calculates the optimal ranges (min, max) for NPK, pH, Temperature, Humidity, and Rainfall
    for a given crop based on the provided dataframe.
    """
    crop_data = dataframe[dataframe['Crop'] == crop_name]

    if crop_data.empty:
        print(f"No data found for crop: {crop_name}")
        return None

    optimal_ranges = {}
    for feature in ['Nitrogen', 'Phosphorus', 'Potassium', 'pH_Value', 'Temperature', 'Humidity', 'Rainfall']:
        optimal_ranges[feature] = {
            'min': crop_data[feature].min(),
            'max': crop_data[feature].max()
        }
    return optimal_ranges

# print("Function 'get_optimal_ranges' defined.")


In [4]:
def recommend_soil_adjustments(current_conditions, optimal_ranges):
    """
    Recommends adjustments to soil and environmental conditions for a crop.
    `current_conditions` should be a dictionary with keys: 'Nitrogen', 'Phosphorus', 'Potassium', 'pH_Value', 'Temperature', 'Humidity', 'Rainfall'.
    `optimal_ranges` should be the output from `get_optimal_ranges`.
    """
    recommendations = []
    for feature in ['Nitrogen', 'Phosphorus', 'Potassium', 'pH_Value', 'Temperature', 'Humidity', 'Rainfall']:
        current_val = current_conditions.get(feature)
        optimal_min = optimal_ranges[feature]['min']
        optimal_max = optimal_ranges[feature]['max']

        if current_val is None:
            recommendations.append(f"Missing current value for {feature}. Optimal range: {optimal_min:.2f} - {optimal_max:.2f}.")
            continue

        if current_val < optimal_min:
            recommendations.append(f"Current {feature} ({current_val:.4f}) is too low. Consider increasing it. Optimal range: {optimal_min:.4f} - {optimal_max:.4f}.")
        elif current_val > optimal_max:
            recommendations.append(f"Current {feature} ({current_val:.4f}) is too high. Consider decreasing it. Optimal range: {optimal_min:.4f} - {optimal_max:.4f}.")
        else:
            recommendations.append(f"Current {feature} ({current_val:.4f}) is within the optimal range ({optimal_min:.4f} - {optimal_max:.4f}).")

    if not recommendations:
        return "No specific recommendations needed based on the provided conditions and optimal ranges."
    else:
        return "\n".join(recommendations)

print("Function 'recommend_soil_adjustments' defined.")

Function 'recommend_soil_adjustments' defined.


In [16]:
#load data
import pandas as pd
df=pd.read_csv(r"D:\researrch\agree.culture.Ai\code\dataset\sensor_Crop_Dataset (1).csv")
df.tail()

,Nitrogen,Phosphorus,Potassium,Temperature,Humidity,pH_Value,Rainfall,Crop,Soil_Type,Variety
19995,15.286598,32.026745,52.276522,30.496937,98.813042,7.549344,238.537544,Rice,Silt,Arborio
19996,29.790472,17.182611,74.772890,40.974020,83.002347,5.895767,333.470901,Wheat,Clay,Durum
19997,25.001919,19.140862,32.719994,29.001299,55.231845,8.230164,119.351274,Wheat,Loamy,Hard Red
19998,74.396171,42.100129,20.669153,22.349399,84.369830,7.878051,385.969414,Wheat,Loamy,Soft Red
19999,41.503429,30.633619,38.022107,15.916907,68.865347,5.044274,326.848992,Potato,Silt,Russet


In [15]:
# Define the target crop for recommendations
target_crop = 'Rice'

# Get optimal ranges for the target crop
rice_optimal_ranges = get_optimal_ranges(target_crop, df)

if rice_optimal_ranges:
    print(f"\nOptimal Ranges for {target_crop}:")
    for feature, ranges in rice_optimal_ranges.items():
        print(f"  {feature}: {ranges['min']:.2f} - {ranges['max']:.2f}")

    # Define example current conditions (you can change these values)
    current_conditions_rice = {
        'Nitrogen': 50,
        'Phosphorus': 80,
        'Potassium': 30,
        'pH_Value': 9,
        'Temperature': 40,
        'Humidity': 85,
        'Rainfall': 250
    }

    print(f"\nCurrent Conditions for {target_crop}:")
    for feature, value in current_conditions_rice.items():
        print(f"  {feature}: {value}")

    # Get recommendations
    recommendations = recommend_soil_adjustments(current_conditions_rice, rice_optimal_ranges)
    print(f"\nRecommendations for {target_crop}:\n{recommendations}")
else:
    print("Could not get optimal ranges for the specified crop.")


Optimal Ranges for Rice:
  Nitrogen: 5.01 - 149.89
  Phosphorus: 5.04 - 89.98
  Potassium: 10.02 - 99.96
  pH_Value: 4.50 - 8.50
  Temperature: 10.00 - 45.00
  Humidity: 30.06 - 99.95
  Rainfall: 20.24 - 399.81

Current Conditions for Rice:
  Nitrogen: 50
  Phosphorus: 80
  Potassium: 30
  pH_Value: 9
  Temperature: 40
  Humidity: 85
  Rainfall: 250

Recommendations for Rice:
Current Nitrogen (50.0000) is within the optimal range (5.0096 - 149.8924).
Current Phosphorus (80.0000) is within the optimal range (5.0410 - 89.9816).
Current Potassium (30.0000) is within the optimal range (10.0234 - 99.9604).
Current pH_Value (9.0000) is too high. Consider decreasing it. Optimal range: 4.5001 - 8.4999.
Current Temperature (40.0000) is within the optimal range (10.0048 - 44.9974).
Current Humidity (85.0000) is within the optimal range (30.0607 - 99.9489).
Current Rainfall (250.0000) is within the optimal range (20.2409 - 399.8129).
